# Checkpoint 3 - Bronze ingestion and silver transformation

Loads Online Retail II CSV data into bronze, transforms it into silver, preserves exceptions, and records quality checks.

In [0]:
from pyspark.sql import functions as F, types as T

source_path = "/Volumes/workspace/default/team9_volume/online_retail_II.csv"
catalog_name = "workspace"
schema_name = "default"
table_prefix = "retail_project"
sample_fraction = 0.01

bronze_table = f"{catalog_name}.{schema_name}.{table_prefix}_bronze"
silver_table = f"{catalog_name}.{schema_name}.{table_prefix}_silver"
exceptions_table = f"{catalog_name}.{schema_name}.{table_prefix}_silver_exceptions"
quality_table = f"{catalog_name}.{schema_name}.{table_prefix}_quality_log"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

DataFrame[]

## 1. Bronze: load raw data as-is

In [0]:
raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("multiLine", True)
    .csv(source_path)
)

# Unity Catalog does not allow spaces in Delta column names
raw = raw.withColumnRenamed("Customer ID", "Customer_ID")

if sample_fraction < 1.0:
    raw = raw.sample(
        withReplacement=False,
        fraction=sample_fraction,
        seed=42
    )

bronze = (
    raw
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit(source_path))
)

bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(bronze_table)

bronze_count = bronze.count()
print(f"Bronze rows: {bronze_count}")
display(bronze.limit(10))

Bronze rows: 10634


Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,_ingested_at,_source_file
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,2026-11-07T04:12:33.210Z,/Volumes/workspace/default/team9_volume/online_retail_II.csv
489462,90082D,DIAMANTE BOW BROOCH BLACK COLOUR,2,2009-12-01 10:49:00,6.35,17592,United Kingdom,2026-11-07T04:12:33.210Z,/Volumes/workspace/default/team9_volume/online_retail_II.csv
489465,21733,RED HANGING HEART T-LIGHT HOLDER,20,2009-12-01 10:52:00,2.95,13767,United Kingdom,2026-11-07T04:12:33.210Z,/Volumes/workspace/default/team9_volume/online_retail_II.csv
489488,84031B,CHARLIE LOLA BLUE HOT WATER BOTTLE,1,2009-12-01 10:59:00,4.65,17238,United Kingdom,2026-11-07T04:12:33.210Z,/Volumes/workspace/default/team9_volume/online_retail_II.csv
489488,21490,SET OF THREE 50'S GIFT WRAPS,3,2009-12-01 10:59:00,1.95,17238,United Kingdom,2026-11-07T04:12:33.210Z,/Volumes/workspace/default/team9_volume/online_retail_II.csv
489519,90024B,NEW BAROQUE B'FLY NECKLACE RED,2,2009-12-01 11:37:00,8.5,17700,United Kingdom,2026-11-07T04:12:33.210Z,/Volumes/workspace/default/team9_volume/online_retail_II.csv
489533,21756,BATH BUILDING BLOCK WORD,1,2009-12-01 12:02:00,5.95,16011,United Kingdom,2026-11-07T04:12:33.210Z,/Volumes/workspace/default/team9_volume/online_retail_II.csv
489536,84671B,CROCHET BEAR WITH BLUE STRIPES,1,2009-12-01 12:13:00,2.55,16393,United Kingdom,2026-11-07T04:12:33.210Z,/Volumes/workspace/default/team9_volume/online_retail_II.csv
489550,21612,SET 12 WILDFLOWER BOTANICAL T-LIGHT,6,2009-12-01 12:33:00,2.95,14156,EIRE,2026-11-07T04:12:33.210Z,/Volumes/workspace/default/team9_volume/online_retail_II.csv
489550,85049A,TRADITIONAL CHRISTMAS RIBBONS,12,2009-12-01 12:33:00,1.25,14156,EIRE,2026-11-07T04:12:33.210Z,/Volumes/workspace/default/team9_volume/online_retail_II.csv


## 2. Silver: standardise, type, deduplicate, and classify records

In [0]:
silver_base = (
    bronze
    .withColumnRenamed("Invoice", "invoice_id")
    .withColumnRenamed("StockCode", "stock_code")
    .withColumnRenamed("Description", "description")
    .withColumnRenamed("Quantity", "quantity")
    .withColumnRenamed("InvoiceDate", "invoice_date")
    .withColumnRenamed("Price", "unit_price")
    .withColumnRenamed("Customer_ID", "customer_id")
    .withColumnRenamed("Country", "country")
    .withColumn("invoice_id", F.trim(F.col("invoice_id")))
    .withColumn("stock_code", F.trim(F.col("stock_code")))
    .withColumn("description", F.trim(F.col("description")))
    .withColumn("country", F.trim(F.col("country")))
    .withColumn("quantity", F.col("quantity").cast(T.IntegerType()))
    .withColumn("unit_price", F.col("unit_price").cast(T.DecimalType(18, 4)))
    .withColumn("customer_id", F.col("customer_id").cast(T.StringType()))
    .withColumn("invoice_date", F.to_timestamp("invoice_date"))
)

deduped = silver_base.dropDuplicates([
    "invoice_id",
    "stock_code",
    "quantity",
    "invoice_date",
    "unit_price",
    "customer_id"
])

silver = (
    deduped
    .withColumn(
        "is_cancelled",
        F.upper(F.col("invoice_id")).startswith("C")
    )
    .withColumn(
        "line_value",
        F.col("quantity") * F.col("unit_price")
    )
    .withColumn(
        "has_exception",
        F.col("invoice_id").isNull()
        | F.col("stock_code").isNull()
        | F.col("invoice_date").isNull()
        | F.col("quantity").isNull()
        | F.col("unit_price").isNull()
        | (F.col("quantity") <= 0)
        | (F.col("unit_price") <= 0)
    )
)

exceptions = silver.filter(F.col("has_exception"))
clean_silver = silver.filter(~F.col("has_exception"))

clean_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(silver_table)

exceptions.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(exceptions_table)

print(
    f"Silver rows: {clean_silver.count()} | "
    f"Exceptions: {exceptions.count()}"
)

display(clean_silver.limit(10))

Silver rows: 10353 | Exceptions: 273


invoice_id,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,_ingested_at,_source_file,is_cancelled,line_value,has_exception
489462,90082D,DIAMANTE BOW BROOCH BLACK COLOUR,2,2009-12-01T10:49:00.000Z,6.3500,17592,United Kingdom,2026-11-07T04:12:44.502Z,/Volumes/workspace/default/team9_volume/online_retail_II.csv,false,12.7000,false
489571,22297,HEART IVORY TRELLIS SMALL,8,2009-12-01T13:29:00.000Z,1.2500,15967,United Kingdom,2026-11-07T04:12:44.502Z,/Volumes/workspace/default/team9_volume/online_retail_II.csv,false,10.0000,false
489607,21558,SKULL LUNCHBOX WITH CUTLERY,1,2009-12-01T14:49:00.000Z,5.1700,null,United Kingdom,2026-11-07T04:12:44.502Z,/Volumes/workspace/default/team9_volume/online_retail_II.csv,false,5.1700,false
489792,21843,RETRO SPOT CAKE STAND,1,2009-12-02T12:06:00.000Z,10.9500,14527,United Kingdom,2026-11-07T04:12:44.502Z,/Volumes/workspace/default/team9_volume/online_retail_II.csv,false,10.9500,false
489815,21821,GLITTER STAR GARLAND WITH BELLS,6,2009-12-02T13:10:00.000Z,3.7500,17909,United Kingdom,2026-11-07T04:12:44.502Z,/Volumes/workspace/default/team9_volume/online_retail_II.csv,false,22.5000,false
490128,21726,MULTI HEARTS STICKERS,6,2009-12-03T18:23:00.000Z,0.8500,15884,United Kingdom,2026-11-07T04:12:44.502Z,/Volumes/workspace/default/team9_volume/online_retail_II.csv,false,5.1000,false
490340,72267,PREMIUM CHURCH CANDLE,6,2009-12-04T15:40:00.000Z,2.1000,13174,United Kingdom,2026-11-07T04:12:44.502Z,/Volumes/workspace/default/team9_volume/online_retail_II.csv,false,12.6000,false
490941,21486,PINK HEART DOTS HOT WATER BOTTLE,24,2009-12-08T14:03:00.000Z,1.6500,14527,United Kingdom,2026-11-07T04:12:44.502Z,/Volumes/workspace/default/team9_volume/online_retail_II.csv,false,39.6000,false
492585,72815,3 WICK CHRISTMAS BRIAR CANDLE,3,2009-12-17T15:26:00.000Z,3.7500,14085,United Kingdom,2026-11-07T04:12:44.502Z,/Volumes/workspace/default/team9_volume/online_retail_II.csv,false,11.2500,false
496579,84997C,BLUE 3 PIECE MINI DOTS CUTLERY SET,2,2010-02-02T13:48:00.000Z,3.7500,12748,United Kingdom,2026-11-07T04:12:44.502Z,/Volumes/workspace/default/team9_volume/online_retail_II.csv,false,7.5000,false


## 3. Repeatable quality checks

In [0]:
required_columns = {
    "invoice_id",
    "stock_code",
    "quantity",
    "invoice_date",
    "unit_price",
    "customer_id",
    "country"
}

actual_columns = set(clean_silver.columns)

# Count duplicate rows removed during silver preparation
dedupe_count = silver_base.count() - deduped.count()

checks = [
    (
        "required_columns_present",
        required_columns.issubset(actual_columns),
        f"missing={sorted(required_columns - actual_columns)}"
    ),
    (
        "silver_has_no_duplicate_business_rows",
        deduped.dropDuplicates([
            "invoice_id",
            "stock_code",
            "quantity",
            "invoice_date",
            "unit_price",
            "customer_id"
        ]).count() == deduped.count(),
        ""
    ),
    (
        "silver_has_positive_quantity",
        clean_silver.filter(F.col("quantity") <= 0).count() == 0,
        ""
    ),
    (
        "silver_has_positive_unit_price",
        clean_silver.filter(F.col("unit_price") <= 0).count() == 0,
        ""
    ),
    (
        "row_reconciliation",
        bronze_count == clean_silver.count() + exceptions.count() + dedupe_count,
        (
            f"bronze={bronze_count}, "
            f"silver={clean_silver.count()}, "
            f"exceptions={exceptions.count()}, "
            f"duplicates_removed={dedupe_count}"
        )
    )
]

quality_log = spark.createDataFrame(
    [
        (name, bool(passed), details)
        for name, passed, details in checks
    ],
    ["check_name", "passed", "details"]
).withColumn(
    "checked_at",
    F.current_timestamp()
)

quality_log.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(quality_table)

display(quality_log)

failed = [
    name
    for name, passed, _ in checks
    if not passed
]

if failed:
    raise ValueError(f"Quality checks failed: {failed}")

print("All Checkpoint 3 quality checks passed.")

check_name,passed,details,checked_at
required_columns_present,true,missing=[],2026-11-07T04:14:52.027Z
silver_has_no_duplicate_business_rows,true,,2026-11-07T04:14:52.027Z
silver_has_positive_quantity,true,,2026-11-07T04:14:52.027Z
silver_has_positive_unit_price,true,,2026-11-07T04:14:52.027Z
row_reconciliation,true,"bronze=10634, silver=10353, exceptions=273, duplicates_removed=8",2026-11-07T04:14:52.027Z


All Checkpoint 3 quality checks passed.
